<a href="https://colab.research.google.com/github/sreent/data-management-intro/blob/main/Document%20Modelling/MongoDB%20Hand-On%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1 Setting Up MongoDB Environment

Run the following cells to install MongoDB on this Colab runtime, install the `cellspell` Jupyter magic so we can write `%%mongodb` cells, and connect to a local database.

**Cell 1 — Install MongoDB:**

In [ ]:
# Install MongoDB on the Colab runtime
!wget -q http://archive.ubuntu.com/ubuntu/pool/main/o/openssl/libssl1.1_1.1.1f-1ubuntu2_amd64.deb
!dpkg -i libssl1.1_1.1.1f-1ubuntu2_amd64.deb > /dev/null 2>&1
!wget -qO - https://www.mongodb.org/static/pgp/server-4.4.asc | apt-key add - > /dev/null 2>&1
!echo "deb [ arch=amd64,arm64 ] http://repo.mongodb.org/apt/ubuntu bionic/mongodb-org/4.4 multiverse" | tee /etc/apt/sources.list.d/mongodb-org-4.4.list > /dev/null
!apt-get update -qq > /dev/null
!apt-get install -y -qq mongodb-org > /dev/null
!mkdir -p /data/db
!mongod --fork --logpath /var/log/mongodb.log --dbpath /data/db
!mongo --quiet --eval 'print("MongoDB ready!")'


**Cell 2 — Install cellspell and load the `%%mongodb` magic:**

In [ ]:
!pip install "cellspell[mongodb] @ git+https://github.com/sreent/jupyter-query-magics.git" -q
%load_ext cellspell.mongodb


**Cell 3 — Connect to the `films` database:**

In [ ]:
%mongodb mongodb://localhost:27017/films


**Cell 3b — Reset the database (safe to re-run):**

In [ ]:
%%mongodb
db.dropDatabase()


# 2 Preparations

Databases and collections in MongoDB are created implicitly while data is inserted. In this tutorial, you will create a collection of *films*. There is no collection so far, so create one by inserting a document.

**Insert your first film — Star Trek Into Darkness:**

In [ ]:
%%mongodb
// Insert Star Trek Into Darkness here


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.insert({
    "title": "Star Trek Into Darkness",
    "year": 2013,
    "genre": [
        "Action",
        "Adventure",
        "Sci-Fi",
    ],
    "actors": [
        "Pine, Chris",
        "Quinto, Zachary",
        "Saldana, Zoe",
    ],
    "releases": [
        {
            "country": "USA",
            "date": ISODate("2013-05-17"),
            "prerelease": true
        },
        {
            "country": "Germany",
            "date": ISODate("2003-05-16"),
            "prerelease": false
        }
    ]
})
```

</details>

Now, there is a *films* collection. You can list the contents of the newly created collection by calling the <code>find()</code> function.

In [ ]:
%%mongodb
// List all films


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find()
```

</details>

If you prefer your result nicely formatted, use <code>pretty()</code>:

In [ ]:
%%mongodb
// Pretty-print the result


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find().pretty()
```

</details>

As you can see, now there is an <code>_id</code> field which is unique for every document

Now insert some more films:

**Insert *Iron Man 3*:**

In [ ]:
%%mongodb
// Insert Iron Man 3


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.insert({
    "title": "Iron Man 3",
    "year": 2013,
    "genre": [
        "Action",
        "Adventure",
        "Sci-Fi",
    ],
    "actors": [
        "Downey Jr., Robert",
        "Paltrow, Gwyneth",
    ]
})
```

</details>

**Insert *This Means War*:**

In [ ]:
%%mongodb
// Insert This Means War


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.insert({
    "title": "This Means War",
    "year": 2011,
    "genre": [
        "Action",
        "Comedy",
        "Romance",
    ],
    "actors": [
        "Pine, Chris",
        "Witherspoon, Reese",
        "Hardy, Tom",
    ],
    "releases": [
        {
            "country": "USA",
            "date": ISODate("2011-02-17"),
            "prerelease": false
        },
        {
            "country": "UK" ,
            "date": ISODate("2011-03-01"),
            "prerelease": true
        }
    ]
})
```

</details>

**Insert *The Amazing Spider - Man 2*:**

In [ ]:
%%mongodb
// Insert Spider-Man 2


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.insert({
    "title": "The Amazing Spider - Man 2",
    "year": 2014,
    "genre": [
        "Action",
        "Adventure",
        "Fantasy",
    ],
    "actors": [
        "Stone, Emma" ,
        "Woodley, Shailene"
    ]
})
```

</details>

# 3 Querying

Now query your collection! Have MongoDB return all films with title **"Iron Man 3"** by calling:

In [ ]:
%%mongodb
// Find films with title 'Iron Man 3'


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({"title": "Iron Man 3"})
```

</details>

Using <code>findOne</code> instead of find produces at most one result (in pretty format):

In [ ]:
%%mongodb
// Use findOne for at most one result


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.findOne({"title": "Iron Man 3"})
```

</details>

Regular expressions can also be used to query a collection. In this tutorial, a short notation is used where the actual regular expression is bounded by slashes (/). The following call yields all movies that start with the letter T:

In [ ]:
%%mongodb
// Find titles starting with T using a /regex/ literal


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({"title": /^T/})
```

</details>

If you are only interested in certain attributes, you can use projection to thin out the produced result. While the selection criteria are given by the first argument of find, the projection is given by the second argument. An example:

In [ ]:
%%mongodb
// Project only the title field


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
  "title": /^T/
},
{
  "title": 1
})
```

</details>

By default, the <code>_id</code> is part of the output, so you have to explicitly suppress it, if you don’t want to have it returned by MongoDB:

In [ ]:
%%mongodb
// Project title and suppress _id


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
  "title": /^T/
},
{
  "_id": 0,
  "title": 1
})
```

</details>

You can also use conditional operators, for example to perform range queries. The following returns the titles of all films starting with the letter T where the year attribute is greater than 2009 and less than or equal to 2011:

In [ ]:
%%mongodb
// Range query: titles starting with T released in 2009 < year ≤ 2011


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
    "year": {
        "$gt": 2009,
        "$lte": 2011
    },
    "title": /^T/
},
{
    "_id": 0,
    "title": 1,
    "year": 1
})
```

</details>

For a logical disjunction of the selection criteria, use the <code>$or</code> operator:

In [ ]:
%%mongodb
// Use $or to combine two criteria


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
    $or: [
      {
          "year": {
              "$gt": 2009,
              "$lte": 2011
          }
      },
      {
          "title": /^T/
      }
    ]
},
{
    "_id": 0,
    "title": 1,
    "year": 1

})
```

</details>

There are also some options that can be appended to the regular expression, e.g. i to achieve case-insensitivity. The following call returns the titles of all movies whose title contains lowercase t, ...

In [ ]:
%%mongodb
// Match titles containing lowercase t


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({"title": /t/}, {"_id": 0, "title": 1})
```

</details>

... whereas the following call also returns titles that contain a T (uppercase):

In [ ]:
%%mongodb
// Add the i flag for case-insensitive match


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
  "title": /t/i
},
{
  "_id": 0,
  "title": 1
})
```

</details>

You can query for exact matches in lists, ...

In [ ]:
%%mongodb
// Films with the Adventure genre


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
  "genre": "Adventure"
},
{
  "_id": 0,
  "title": 1,
  "genre": 1
})
```

</details>

... but you can also query for partial matches which yields all genres that start with the letter A:

In [ ]:
%%mongodb
// Films with a genre starting with A


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
  "genre": /^A/
},
{
  "_id": 0,
  "title": 1,
  "genre": 1
})
```

</details>

There are also more complex operators for more complex selection criteria, e.g. the <code>$all</code> operator. The following call prints the title and actors of every movie for which each of two given regular expressions matches at least one of its actors:

In [ ]:
%%mongodb
// Use $all with two regex literals


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
    "actors": {
      "$all": [/pine/i, /zachary/i]
    }
},
{
    "_id": 0,
    "title": 1,
    "actors": 1
})
```

</details>

In contrast, the <code>$nin</code>, i.e. not in, operator checks for the lack of matching values, i.e. actor names that do not match either one of the given regular expressions:

In [ ]:
%%mongodb
// Use $nin to exclude actor matches


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
    "actors": {
      $nin: [/pine/i, /zachary/i]
    }
},
{
    "_id": 0,
    "title": 1,
    "actors": 1
})
```

</details>

The <code>$exists</code> operator can be used to check for the existence of an attribute, e.g. to select only movies with undefined releases:

In [ ]:
%%mongodb
// Films with no releases attribute


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
  releases: {$exists: false}
},
{
  "_id": 0,
  "title": 1
})
```

</details>

In MongoDB, it is also possible to query nested data, i.e. subdocuments. The following returns the title and releases of every movie that is known to be released in the UK:

In [ ]:
%%mongodb
// Films released in the UK (dot notation)


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
  "releases.country": "UK"
},
{
  "_id": 0,
  "title": 1,
  "releases": 1
})
```

</details>

Please note that you have to use quotes to address nested fields.

Applying more complex selection criteria on a nested document, however, is a little tricky. For example, if you wanted MongoDB to return all movies that had their prerelease in the USA, you might try something like this:

In [ ]:
%%mongodb
// Films with USA prerelease (naive — see warning below)


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
    "releases.country": "USA" ,
    "releases.prerelease": true
},
{
    "_id": 0 ,
    "title": 1,
    "releases": 1
})
```

</details>

However, This Means War is also returned, but was prereleased in the UK. The call above actually returns all movies that have some prerelease or were released in the USA. To only select movies were both applies to the same release, the <code>$elemMatch</code> can be used:



In [ ]:
%%mongodb
// Use $elemMatch so both conditions match the same release


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
  releases: {
    $elemMatch: {
      country: "USA",
      prerelease: true
    }
  }
})
```

</details>

Naturally, there are many other operators not covered by this tutorial.

# 4 Update

You can also add or update fields in a document by using the <code>$set</code> operator. For example, you can add a rating field to one of the movies:

In [ ]:
%%mongodb
// Add a rating field to Star Trek Into Darkness


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.update(
    {"title": "Star Trek Into Darkness"},
    {$set: {"rating": 6.4}}
)
```

</details>

If you do not use the $set operator, every document fulfilling the selection criteria will be replaced, so be careful!

Now, verify if the <code>rating</code> field is added to the document:

In [ ]:
%%mongodb
// Verify the rating field was added


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
  "title": "Star Trek Into Darkness"
})
```

</details>

To increment a number of value, you can use the <code>$inc</code> operator:

In [ ]:
%%mongodb
// Increment the rating using $inc


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.update(
    {"title": "Star Trek Into Darkness"},
    {$inc: {"rating": 0.1}}
)
```

</details>

Verify if the rating value has been incremented by <code>0.1</code>.

In [ ]:
%%mongodb
// Verify the rating was incremented


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({
  "title": "Star Trek Into Darkness"
})
```

</details>

Again, there are many other different operators for different purposes, e.g. `$unset`, `$inc`, `$pop`, `$push`, `$pushAll` or `$addToSet`.

# 5 Delete

You can remove documents with the remove function. It actually works almost like the find function; you only don’t use the projection parameter. If, for example, you want to remove all film documents whose title starts with the letter T, you can first query for all such movies...

In [ ]:
%%mongodb
// Find films whose title starts with T


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find({"title": /^T/})
```

</details>

... to verify that your selection criteria is correct and then replaced the find in your call by remove:

In [ ]:
%%mongodb
// Remove films whose title starts with T


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.remove({"title": /^T/})
```

</details>

Now, we verify if the documents has been removed from the collection:

In [ ]:
%%mongodb
// Verify the documents were removed


<details>
<summary><strong>Click to reveal solution</strong></summary>

```js
%%mongodb
db.films.find()
```

</details>